# TAKTKRONE-I Quickstart Guide

Get up and running with TAKTKRONE-I in 5 minutes!

## Installation

Install TAKTKRONE-I with all dependencies:

In [ ]:
# Install from source
!git clone https://github.com/anthropics/taktkrone-i.git
!cd taktkrone-i && pip install -e .[serve]


## Load a Pretrained Model

Load a model for inference:

In [ ]:
from occlm.serving.engine import AsyncOCCInferenceEngine
import asyncio

# Initialize engine
engine = AsyncOCCInferenceEngine(
    model_name="taktkrone-v0.1-sft-baseline",
    device="cuda"  # or "cpu"
)

# Load model
asyncio.run(engine.load())
print("Model loaded successfully!")
print(engine.get_model_info())


## Run Single Query

Query the model for a single incident:

In [ ]:
query = """Signal failure detected at Station A on Line 1 northbound. 
What actions should the OCC take?"""

# Single inference
result = asyncio.run(engine.query_async(
    query=query,
    max_tokens=256,
    temperature=0.7
))

print("Query:", query)
print("\nResponse:")
print(result['response'])
print(f"\nLatency: {result['latency_ms']:.1f}ms")
print(f"Confidence: {result['confidence']:.2f}")


## Batch inference

Process multiple queries efficiently:

In [ ]:
queries = [
    "Power outage at Station B affecting Line 2",
    "Medical emergency on northbound train",
    "Overcrowding at peak hours affecting service"
]

# Batch inference
results = asyncio.run(engine.batch_query_async(queries, max_tokens=200))

for i, (query, result) in enumerate(zip(queries, results), 1):
    print(f"\n--- Query {i} ---")
    print(f"Input: {query}")
    print(f"Output: {result['response'][:150]}...")
    print(f"Time: {result['latency_ms']:.1f}ms")


## API Server

Start the OCC API server:

In [ ]:
# Start server from command line:
# occlm serve api --model-path taktkrone-v0.1-sft-baseline --port 8000
# 
# Then query the API:
import requests

response = requests.post(
    "http://localhost:8000/v1/query",
    json={
        "query": "Signal failure at Station A. What should OCC do?",
        "operator": "mta_nyct",
        "max_tokens": 256
    }
)

result = response.json()
print("API Response:")
print(result)


## Interactive Demo with Gradio

Launch the interactive UI:

In [ ]:
# Launch interactive UI:
# occlm serve demo --model-path taktkrone-v0.1 --port 7860
#
# Then open http://localhost:7860 in your browser

print("To start the interactive demo UI:")
print("  occlm serve demo --model-path taktkrone-v0.1-sft-baseline")
print("\nThen open: http://localhost:7860")


## Next Steps

- See `training.ipynb` to fine-tune on your own data
- See `evaluation.ipynb` to run benchmarks
- Check [documentation](../docs/) for more details
